# 경로 탐색과 개인화 실행

설치 후 이 노트북에서 입력을 바꾸어 실행합니다. 웹 서버는 사용하지 않습니다. `integrated`는 통합, `transit`은 버스·지하철, `bicycle`은 따릉이입니다. API 호출에는 기존 안전한도가 적용됩니다.


In [ ]:
from pathlib import Path
import sys
roots=[Path.cwd(),*Path.cwd().parents]
candidates=[q for p in roots for q in (p,p/'bicycle')]
ROOT=next((p for p in candidates if (p/'src/project_service.py').is_file()),None)
if ROOT is None:raise FileNotFoundError('bicycle 프로젝트 폴더에서 노트북을 실행해 주세요.')
sys.path.insert(0,str(ROOT/'src'))
from project_service import ProjectService


In [ ]:
ORIGIN={'name':'서울역','latitude':37.5568,'longitude':126.9723}
DESTINATION={'name':'여의도 한강공원','latitude':37.5284,'longitude':126.9326}
DEPARTURE='auto'  # 또는 시간대 포함 미래 시각
MODE='transit'
PROFILE_ID=None  # 등록 이력이 있는 개인 ID를 선택적으로 입력


In [ ]:
with ProjectService() as service:
    result=service.find_routes(ORIGIN,DESTINATION,departure=DEPARTURE,mode=MODE,profile_id=PROFILE_ID)
print(result['status'],result['termination_reason'])
for i,route in enumerate(result['routes'],1):
    print(i,round(route['total_minutes'],2),[leg['mode'] for leg in route['legs']])


## 사진 등록

필요한 경우 아래 예제를 별도 셀에서 실행합니다. `preview_screenshot` 결과와 원본을 대조한 뒤 `confirm_screenshot`으로 등록합니다. 사진 파일은 저장하지 않고 추출 이력만 ID별 DB에 저장합니다.

```python
with ProjectService() as service:
    profile_id = service.create_profile()["profile_id"]
    draft = service.preview_screenshot(profile_id, "C:/photos/ride.png")
    print(draft)

# 결과 확인 후 실행
with ProjectService() as service:
    saved = service.confirm_screenshot(profile_id, draft["draft_id"], 0, confirmed=True)
    print(service.get_profile(profile_id))
```
